<a href="https://colab.research.google.com/github/JeyssonP92/Proyectos-Python-y-QA/blob/main/pipeline_end_to_end_data_qa_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline End-to-End de Data QA: Contratos de Datos, SQLite y Dashboard Interactivo

Este proyecto implementa un flujo automatizado de ingeniería y calidad de datos (**ETL + Data QA**) que audita, repara, almacena y visualiza registros de clientes antes de su paso a producción.

## Arquitectura del Proyecto

1. **Validación de Esquemas y Regex (`Pandera`):** Inspecciona los datos entrantes aplicando reglas estrictas de negocio (IDs positivos) y expresiones regulares para validar formatos de correo electrónico, teléfonos de 10 dígitos y códigos postales.
2. **Auto-reparación y Cuarentena (`Dead Letter Queue`):** Corrige automáticamente errores predecibles (rellenando códigos postales incompletos con ceros a la izquierda) y aísla los registros con fallos críticos en una tabla separada sin detener la operación.
3. **Almacenamiento Relacional (`SQLite`):** Persiste los datos limpios (`clientes_aprobados`) y los registros rechazados (`clientes_cuarentena`) en una base de datos SQL para su consulta y auditoría.
4. **Certificación Automatizada (`Pytest / ipytest`):** Ejecuta pruebas unitarias directamente sobre las tablas SQLite para garantizar la integridad referencial y el cumplimiento del contrato antes de exponer los datos.
5. **Dashboard Ejecutivo Interactivo (`Plotly` & `ipywidgets`):** Conecta directamente con SQLite para ofrecer un panel visual en tiempo real.

## Qué muestra el Dashboard

* **Monitor de Salud de Datos (Gráfico de anillo):** Resume visualmente el porcentaje de calidad del lote procesado, contrastando los registros aprobados para producción (60% en verde) frente a los registros bloqueados en cuarentena (40% en rojo).
* **Inspector Interactivo de SQLite:** Panel de consulta en vivo con filtro desplegable (`Ver Tabla`) que permite auditar al instante tanto los clientes certificados como los registros aislados para revisión manual sin escribir consultas SQL.

## Tecnologías Utilizadas

* **Lenguaje:** Python 3
* **Manipulación y Validación:** Pandas, Pandera, Regex
* **Base de Datos:** SQLite3
* **Testing / QA:** Pytest (`ipytest`)
* **Visualización e Interactividad:** Plotly Express, ipywidgets

In [4]:
# Se instala Pandera para contratos de datos, Pandas e ipytest para ejecutar pruebas dentro del cuaderno
!pip install pandera pandas ipytest plotly -q

import pandas as pd
import pandera.pandas as pa
from pandera import Column, DataFrameSchema, Check
import ipytest
import sqlite3
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

# ipytest para ejecutar pruebas dentro del cuaderno
ipytest.autoconfig()



Se creo un conjunto de datos de registro de clientes que simula lo que llega desde un formulario web o una API externa. Incluye registros sanos y 4 errores intencionales escondidos.

In [5]:
# Se hace una simulacion de un lote de registros de clientes entrantes
datos_clientes = {
    "id_cliente": [101, 102, -5, 104, 105],  # Error oculto: ID negativo (-5)
    "nombre": ["Ana Gómez", "Carlos Ruiz", "Laura Díaz", "Pedro Pérez", "Sofía Mora"],
    "correo": [
        "ana.gomez@empresa.com",
        "carlos.ruiz@correo.co",
        "laura.diaz_sin_arroba.com",  # Error oculto: falta el @
        "pedro@dominio.org",
        "sofia@empresa.com"
    ],
    "telefono": [
        "3001234567",
        "3109876543",
        "3205551234",
        "315ABC4567",  # Error oculto: contiene letras en el teléfono
        "3014448899"
    ],
    "codigo_postal": [
        "110111",
        "050001",
        "760001",
        "080001",
        "123"  # Error oculto: menos de 6 dígitos
    ]
}

df_clientes = pd.DataFrame(datos_clientes)

print("VISTA PREVIA DE LOS DATOS ENTRANTES SIN VALIDAR")
print(df_clientes)

VISTA PREVIA DE LOS DATOS ENTRANTES SIN VALIDAR
   id_cliente       nombre                     correo    telefono  \
0         101    Ana Gómez      ana.gomez@empresa.com  3001234567   
1         102  Carlos Ruiz      carlos.ruiz@correo.co  3109876543   
2          -5   Laura Díaz  laura.diaz_sin_arroba.com  3205551234   
3         104  Pedro Pérez          pedro@dominio.org  315ABC4567   
4         105   Sofía Mora          sofia@empresa.com  3014448899   

  codigo_postal  
0        110111  
1        050001  
2        760001  
3        080001  
4           123  


Se construye el validador y la prueba Automatizada

In [6]:
# Se define el Contrato de Datos estricto usando Pandera y Regex
esquema_clientes = DataFrameSchema({
    "id_cliente": Column(
        int,
        checks=Check.greater_than(0, error="El ID debe ser un número positivo mayor a 0.")
    ),
    "nombre": Column(
        str,
        nullable=False
    ),
    "correo": Column(
        str,
        checks=Check.str_matches(
            r"^[\w\.-]+@[\w\.-]+\.\w+$",
            error="Formato de correo electrónico inválido."
        )
    ),
    "telefono": Column(
        str,
        checks=Check.str_matches(
            r"^\d{10}$",
            error="El teléfono debe tener exactamente 10 dígitos numéricos."
        )
    ),
    "codigo_postal": Column(
        str,
        checks=Check.str_matches(
            r"^\d{6}$",
            error="El código postal debe tener exactamente 6 dígitos."
        )
    )
})

# Se ejecuta la auditoría en modo 'lazy=True' para listar TODOS los fallos juntos

print("Ejecutando auditoría de contrato de datos con Pandera \n")
try:
    esquema_clientes.validate(df_clientes, lazy=True)
    print("✅ ¡Contrato aprobado! Todos los datos son válidos.")
except pa.errors.SchemaErrors as err:
    print("¡ALERTA DE DATA QA! Se rechazó el lote por los siguientes errores:")
    # Se muestra la tabla resumida de errores detectados por columna y fila
    print(err.failure_cases[["column", "check", "failure_case", "index"]])

Ejecutando auditoría de contrato de datos con Pandera 

¡ALERTA DE DATA QA! Se rechazó el lote por los siguientes errores:
          column                                              check  \
0     id_cliente       El ID debe ser un número positivo mayor a 0.   
1         correo            Formato de correo electrónico inválido.   
2       telefono  El teléfono debe tener exactamente 10 dígitos ...   
3  codigo_postal  El código postal debe tener exactamente 6 dígi...   

                failure_case  index  
0                         -5      2  
1  laura.diaz_sin_arroba.com      2  
2                 315ABC4567      3  
3                        123      4  


se auto-reparan los datos faltantes o que estan mal, y/o se dejan aislados los datos criticos que no se pueden asumir.

In [8]:
print("Iniciando proceso de auto reparacion y se aislan los datos faltantes\n")
df_procesado = df_clientes.copy()

# Auto-reparacion, se copleta la informacion del codigo postal con ceros a la izquierda usando .zfill(6) para que quede 000123)
df_procesado["codigo_postal"] = df_procesado["codigo_postal"].apply(
    lambda x: x.zfill(6) if x.isdigit() and len(x) < 6 else x
)
try:
    esquema_clientes.validate(df_procesado, lazy=True)
    df_aprobados = df_procesado
    df_cuarentena = pd.DataFrame()
except pa.errors.SchemaErrors as err:
    # Obtenemos los números de fila (índices) que siguen teniendo errores graves
    filas_con_error = err.failure_cases["index"].unique()

    # Aislamos los registros dañados en la tabla de cuarentena
    df_cuarentena = df_procesado.loc[filas_con_error]

    # Dejamos pasar únicamente los registros limpios al flujo principal
    df_aprobados = df_procesado.drop(index=filas_con_error)

print("Registros aprobados para produccion (Incluye el código postal reparado):")
print(df_aprobados)

Iniciando proceso de auto reparacion y se aislan los datos faltantes

Registros aprobados para produccion (Incluye el código postal reparado):
   id_cliente       nombre                 correo    telefono codigo_postal
0         101    Ana Gómez  ana.gomez@empresa.com  3001234567        110111
1         102  Carlos Ruiz  carlos.ruiz@correo.co  3109876543        050001
4         105   Sofía Mora      sofia@empresa.com  3014448899        000123


Se guardan las tablas ya separadas dentro de la base de datos SQLite

In [9]:
NOMBRE_BD = "clientes_data_qa.db"
conexion = sqlite3.connect(NOMBRE_BD)

df_aprobados.to_sql("clientes_aprobados", conexion, if_exists="replace", index=False)
df_cuarentena.to_sql("clientes_cuarentena", conexion, if_exists="replace", index=False)
conexion.close()

print(f"\nTablas guardadas con éxito en la base de datos SQL: {NOMBRE_BD}")


Tablas guardadas con éxito en la base de datos SQL: clientes_data_qa.db


Imprimir los registros en aislados y ejecutar pruebas automatizadas con Pytest

In [10]:
print("\nRegistros aislados (Para revisión manual):")
print(df_cuarentena)

def test_contrato_datos_aprobados():
    # Validamos que la tabla aprobada pase el esquema de Pandera sin lanzar ningún error
    esquema_clientes.validate(df_aprobados)
    assert len(df_aprobados) == 3, "Deberían haber pasado exactamente 3 registros limpios."
    assert len(df_cuarentena) == 2, "Deberían haber quedado exactamente 2 registros en cuarentena."

print("\n Ejecutando prueba unitaria final sobre los datos limpios")
ipytest.run()


Registros aislados (Para revisión manual):
   id_cliente       nombre                     correo    telefono  \
2          -5   Laura Díaz  laura.diaz_sin_arroba.com  3205551234   
3         104  Pedro Pérez          pedro@dominio.org  315ABC4567   

  codigo_postal  
2        760001  
3        080001  

 Ejecutando prueba unitaria final sobre los datos limpios
.                                                                                            [100%]
========================================= warnings summary =========================================
../usr/local/lib/python3.13/dist-packages/_pytest/config/__init__.py:1290
  /usr/local/lib/python3.13/dist-packages/_pytest/config/__init__.py:1290: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; typeguard
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
1 passed, 1 warning in 0.02s


<ExitCode.OK: 0>

Se realiza un Dashboard

In [11]:
# Se lee las tablas certificadas directamente desde SQLite
conn = sqlite3.connect("clientes_data_qa.db")
df_prod = pd.read_sql_query("SELECT * FROM clientes_aprobados", conn)
df_error = pd.read_sql_query("SELECT * FROM clientes_cuarentena", conn)
conn.close()

# Gráfica de Anillo: Salud de Calidad de Datos (Aprobados vs Cuarentena)
resumen_qa = pd.DataFrame({
    "Estado": ["Aprobados (Producción)", "Aislados (Cuarentena)"],
    "Registros": [len(df_prod), len(df_error)]
})

fig_salud = px.pie(
    resumen_qa,
    names="Estado",
    values="Registros",
    title="Monitor de Salud de Datos (Fuente: SQLite)",
    color="Estado",
    color_discrete_map={"Aprobados (Producción)": "#2ecc71", "Aislados (Cuarentena)": "#e74c3c"},
    hole=0.45
)
fig_salud.update_traces(textinfo="value+percent")
fig_salud.update_layout(height=380)
fig_salud.show()

# Panel Interactivo: Inspector de Tablas SQL en Vivo
print("\n🎛️ --- INSPECTOR INTERACTIVO DE BASE DE DATOS SQLITE ---")
selector_tabla = widgets.Dropdown(
    options=["Clientes Aprobados (Producción)", "Clientes en Cuarentena (Errores)"],
    value="Clientes Aprobados (Producción)",
    description="Ver Tabla:"
)
salida_tabla = widgets.Output()

def mostrar_tabla_sql(change):
    with salida_tabla:
        clear_output(wait=True)
        if selector_tabla.value == "Clientes Aprobados (Producción)":
            print(f"✅ Mostrando {len(df_prod)} registros certificados en SQLite:")
            display(df_prod)
        else:
            print(f"🚨 Mostrando {len(df_error)} registros aislados para revisión manual:")
            display(df_error)

selector_tabla.observe(mostrar_tabla_sql, names="value")
display(selector_tabla, salida_tabla)
mostrar_tabla_sql(None)


🎛️ --- INSPECTOR INTERACTIVO DE BASE DE DATOS SQLITE ---


Dropdown(description='Ver Tabla:', options=('Clientes Aprobados (Producción)', 'Clientes en Cuarentena (Errore…

Output()